# Chapitre 5 — Bases vectorielles et architectures de retrieval

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahouahounko/rag-en-pratique/blob/main/chapters/chapitre-05-bases-vectorielles/05_bases_vectorielles.ipynb)

Ce laboratoire transforme les 14 extraits du chapitre en exemples autonomes. Les appels OpenAI et Pinecone sont désactivés par défaut.

## Ressources utiles

- [FAISS](https://faiss.ai/)
- [Qdrant](https://qdrant.tech/documentation/)
- [Chroma](https://docs.trychroma.com/)
- [Pinecone](https://docs.pinecone.io/)
- [OpenAI — embeddings](https://developers.openai.com/api/docs/guides/embeddings)
- [Sentence Transformers — Retrieve & Re-Rank](https://sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html)

## 0. Préparer Colab ou Jupyter

Installe les moteurs utilisés dans le chapitre.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not Path("src").is_dir():
    if not Path("rag-en-pratique").is_dir():
        subprocess.run(["git", "clone", "https://github.com/Ahouahounko/rag-en-pratique.git"], check=True)
    os.chdir("rag-en-pratique")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[vectorstores,vector-services]"],
    check=True,
)
print("Environnement du chapitre 5 prêt :", Path.cwd())


## Configuration OpenAI facultative

Activez-la seulement avant les exemples concernés.

In [ ]:
# @title Activer les exemples OpenAI
UTILISER_OPENAI = False # @param {type:"boolean"}

if UTILISER_OPENAI:
    import os
    import subprocess
    import sys
    from getpass import getpass

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[openai]"], check=True)
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY : ")
    if not os.getenv("OPENAI_MODEL"):
        os.environ["OPENAI_MODEL"] = input("OPENAI_MODEL : ").strip()
    print("OpenAI est activé pour les exemples 4, 7, 8, 10, 11, 13 et 14.")
else:
    print("OpenAI désactivé : les exemples locaux restent exécutables.")


## Configuration Pinecone facultative

La clé est saisie de manière masquée et jamais enregistrée.

In [ ]:
# @title Activer Pinecone pour l'exemple 6
UTILISER_PINECONE = False # @param {type:"boolean"}

if UTILISER_PINECONE:
    import os
    from getpass import getpass

    if not os.getenv("PINECONE_API_KEY"):
        os.environ["PINECONE_API_KEY"] = getpass("PINECONE_API_KEY : ")
    print("Pinecone est activé. La création d'un index distant peut être facturée.")
else:
    print("Pinecone désactivé.")


## 1. FAISS HNSW

Construire un graphe navigable et régler efSearch.

Script correspondant : [`01_faiss_hnsw.py`](examples/01_faiss_hnsw.py)

In [ ]:
# ruff: noqa: F811
"""Construire et interroger un index FAISS HNSW."""

from __future__ import annotations

import numpy as np


def construire_index_hnsw(
    vecteurs: np.ndarray,
    dimension: int,
    m: int = 32,
    ef_construction: int = 200,
):
    """Crée un index HNSW utilisant le produit scalaire sur des vecteurs normalisés."""
    import faiss

    donnees = np.asarray(vecteurs, dtype=np.float32).copy()
    if donnees.ndim != 2 or donnees.shape[1] != dimension:
        raise ValueError(f"Forme attendue : (n, {dimension})")
    faiss.normalize_L2(donnees)
    index = faiss.IndexHNSWFlat(dimension, m, faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = ef_construction
    index.add(donnees)
    return index


def chercher(index, vecteur_requete: np.ndarray, k: int = 5, ef_search: int = 64):
    """Renvoie les scores cosinus et les positions des voisins."""
    import faiss

    index.hnsw.efSearch = ef_search
    requete = np.asarray(vecteur_requete, dtype=np.float32).reshape(1, -1).copy()
    faiss.normalize_L2(requete)
    scores, positions = index.search(requete, k)
    return scores[0], positions[0]


if __name__ == "__main__":
    corpus = np.array([[1, 0, 0], [0.9, 0.1, 0], [0, 1, 0], [0, 0, 1]], dtype=np.float32)
    hnsw = construire_index_hnsw(corpus, dimension=3, m=8)
    scores, ids = chercher(hnsw, np.array([1, 0, 0]), k=2)
    print(list(zip(ids.tolist(), scores.round(3).tolist(), strict=True)))


## 2. FAISS IVF

Entraîner les centroïdes puis choisir le nombre de listes sondées.

Script correspondant : [`02_faiss_ivf.py`](examples/02_faiss_ivf.py)

In [ ]:
# ruff: noqa: F811
"""Construire un index FAISS IVF et régler son compromis rappel/latence."""

from __future__ import annotations

import numpy as np


def construire_index_ivf(
    vecteurs: np.ndarray,
    dimension: int,
    n_list: int | None = None,
    graine: int = 42,
):
    import faiss

    donnees = np.asarray(vecteurs, dtype=np.float32).copy()
    if donnees.ndim != 2 or donnees.shape[1] != dimension:
        raise ValueError(f"Forme attendue : (n, {dimension})")
    n = len(donnees)
    if n < 2:
        raise ValueError("IVF demande au moins deux vecteurs")
    n_list = n_list or max(1, min(int(np.sqrt(n)), n // 2))
    faiss.normalize_L2(donnees)

    quantifieur = faiss.IndexFlatIP(dimension)
    index = faiss.IndexIVFFlat(quantifieur, dimension, n_list, faiss.METRIC_INNER_PRODUCT)
    generateur = np.random.default_rng(graine)
    echantillon = donnees
    if n > 500_000:
        echantillon = donnees[generateur.choice(n, 500_000, replace=False)]
    index.train(echantillon)
    index.add(donnees)
    index.nprobe = max(1, n_list // 10)
    return index


def chercher(index, requete: np.ndarray, k: int = 5, nprobe: int | None = None):
    import faiss

    if nprobe is not None:
        index.nprobe = nprobe
    vecteur = np.asarray(requete, dtype=np.float32).reshape(1, -1).copy()
    faiss.normalize_L2(vecteur)
    return tuple(element[0] for element in index.search(vecteur, k))


if __name__ == "__main__":
    rng = np.random.default_rng(42)
    corpus = rng.normal(size=(200, 8)).astype(np.float32)
    ivf = construire_index_ivf(corpus, dimension=8, n_list=8)
    scores, ids = chercher(ivf, corpus[0], k=3, nprobe=4)
    print(list(zip(ids.tolist(), scores.round(3).tolist(), strict=True)))


## 3. Filtres Qdrant

Restreindre les candidats avant la recherche vectorielle.

Script correspondant : [`03_filtrage_metadonnees.py`](examples/03_filtrage_metadonnees.py)

In [ ]:
# ruff: noqa: F811
"""Appliquer des filtres de métadonnées avant une recherche Qdrant."""

from __future__ import annotations

from typing import Any


def construire_filtre(departement: str | None = None, date_minimale: str | None = None):
    from qdrant_client.models import DatetimeRange, FieldCondition, Filter, MatchValue

    conditions = []
    if departement:
        conditions.append(FieldCondition(key="departement", match=MatchValue(value=departement)))
    if date_minimale:
        conditions.append(FieldCondition(key="date", range=DatetimeRange(gte=date_minimale)))
    return Filter(must=conditions) if conditions else None


def chercher_avec_filtres(
    client: Any,
    collection: str,
    vecteur_requete: list[float],
    departement: str | None = None,
    date_minimale: str | None = None,
    k: int = 5,
) -> list[Any]:
    """Effectue une recherche filtrée avec l'API universelle query_points."""
    resultat = client.query_points(
        collection_name=collection,
        query=vecteur_requete,
        query_filter=construire_filtre(departement, date_minimale),
        limit=k,
        with_payload=True,
    )
    return list(resultat.points)


if __name__ == "__main__":
    from qdrant_client import QdrantClient
    from qdrant_client.models import Distance, PointStruct, VectorParams

    qdrant = QdrantClient(":memory:")
    qdrant.create_collection("documents", vectors_config=VectorParams(size=3, distance=Distance.COSINE))
    qdrant.upsert(
        "documents",
        points=[
            PointStruct(id=1, vector=[1.0, 0.0, 0.0], payload={"departement": "juridique", "date": "2026-01-15", "texte": "Retour sous 30 jours"}),
            PointStruct(id=2, vector=[0.0, 1.0, 0.0], payload={"departement": "finance", "date": "2025-01-15", "texte": "Bilan annuel"}),
        ],
    )
    print([point.payload for point in chercher_avec_filtres(qdrant, "documents", [1, 0, 0], "juridique")])


## 4. Chroma et OpenAI

Persister une collection et calculer les embeddings avec OpenAI.

Script correspondant : [`04_chroma.py`](examples/04_chroma.py)

In [ ]:
# ruff: noqa: F811
"""Indexer et rechercher dans Chroma avec les embeddings OpenAI."""

from __future__ import annotations

import os
from pathlib import Path


def creer_collection(repertoire: str | Path = "./chroma_db"):
    """Crée une collection persistante ; OPENAI_API_KEY doit être définie."""
    import chromadb
    from chromadb.utils import embedding_functions

    cle = os.getenv("OPENAI_API_KEY")
    if not cle:
        raise RuntimeError("Définissez OPENAI_API_KEY pour cet exemple Chroma/OpenAI.")
    fonction = embedding_functions.OpenAIEmbeddingFunction(
        api_key=cle,
        model_name="text-embedding-3-small",
    )
    client = chromadb.PersistentClient(path=str(repertoire))
    return client.get_or_create_collection(
        name="base_documentaire",
        embedding_function=fonction,
        metadata={"hnsw:space": "cosine"},
    )


def indexer_et_chercher(collection) -> dict[str, object]:
    collection.upsert(
        documents=[
            "La garantie standard couvre 24 mois à compter de la livraison.",
            "Le remboursement peut être demandé sous 30 jours.",
        ],
        metadatas=[
            {"source": "cgv.pdf", "page": 8, "departement": "juridique"},
            {"source": "cgv.pdf", "page": 12, "departement": "juridique"},
        ],
        ids=["chunk_001", "chunk_002"],
    )
    return collection.query(
        query_texts=["politique de remboursement"],
        n_results=2,
        where={"departement": "juridique"},
    )


if __name__ == "__main__":
    if not os.getenv("OPENAI_API_KEY"):
        print("Exemple prêt : définissez OPENAI_API_KEY pour l'exécuter.")
    else:
        print(indexer_et_chercher(creer_collection())["documents"])


## 5. Collection Qdrant

Configurer vecteurs denses, sparse et paramètres HNSW.

Script correspondant : [`05_qdrant.py`](examples/05_qdrant.py)

In [ ]:
# ruff: noqa: F811
"""Configurer une collection Qdrant dense et sparse."""

from __future__ import annotations


def creer_collection(client, nom: str, dimension: int = 768) -> None:
    from qdrant_client.models import (
        Distance,
        HnswConfigDiff,
        OptimizersConfigDiff,
        SparseIndexParams,
        SparseVectorParams,
        VectorParams,
    )

    client.create_collection(
        collection_name=nom,
        vectors_config=VectorParams(
            size=dimension,
            distance=Distance.COSINE,
            hnsw_config=HnswConfigDiff(
                m=32,
                ef_construct=200,
                full_scan_threshold=10_000,
            ),
        ),
        sparse_vectors_config={
            "lexical": SparseVectorParams(index=SparseIndexParams(on_disk=False))
        },
        optimizers_config=OptimizersConfigDiff(
            indexing_threshold=20_000,
            memmap_threshold=50_000,
        ),
    )


if __name__ == "__main__":
    from qdrant_client import QdrantClient

    qdrant = QdrantClient(":memory:")
    creer_collection(qdrant, "demo", dimension=3)
    print(qdrant.get_collection("demo").status)


## 6. Pinecone serverless

Créer explicitement un index distant et un namespace.

Script correspondant : [`06_pinecone.py`](examples/06_pinecone.py)

In [ ]:
# ruff: noqa: F811
"""Créer un index Pinecone serverless et insérer un lot de vecteurs."""

from __future__ import annotations

import os
from typing import Any


def connecter_pinecone(api_key: str | None = None):
    from pinecone import Pinecone

    cle = api_key or os.getenv("PINECONE_API_KEY")
    if not cle:
        raise RuntimeError("Définissez PINECONE_API_KEY pour utiliser Pinecone.")
    return Pinecone(api_key=cle)


def preparer_index(
    pc: Any,
    nom: str = "rag-production",
    dimension: int = 1536,
    cloud: str = "aws",
    region: str = "eu-west-1",
):
    from pinecone import ServerlessSpec

    if nom not in pc.list_indexes().names():
        pc.create_index(
            name=nom,
            dimension=dimension,
            metric="cosine",
            spec=ServerlessSpec(cloud=cloud, region=region),
        )
    return pc.Index(nom)


def inserer_passage(index, vecteur: list[float], namespace: str = "finance") -> None:
    index.upsert(
        vectors=[
            {
                "id": "chunk_001",
                "values": vecteur,
                "metadata": {
                    "source": "rapport_annuel_2024.pdf",
                    "page": 15,
                    "departement": "finance",
                    "texte": "Le chiffre d'affaires annuel atteint...",
                },
            }
        ],
        namespace=namespace,
    )


if __name__ == "__main__":
    if not os.getenv("PINECONE_API_KEY"):
        print("Exemple prêt : définissez PINECONE_API_KEY pour créer l'index distant.")
    else:
        pinecone = connecter_pinecone()
        print(preparer_index(pinecone))


## 7. Self-Query

Séparer le texte recherché des filtres structurés.

Script correspondant : [`07_self_query.py`](examples/07_self_query.py)

In [ ]:
# ruff: noqa: F811
"""Transformer une question en requête sémantique et filtres structurés."""

from __future__ import annotations

import json
import os
from dataclasses import dataclass
from typing import Any


@dataclass(frozen=True)
class RequeteStructuree:
    texte: str
    filtres: dict[str, str]


SCHEMA = {
    "departement": ["RH", "juridique", "finance", "produit", "IT"],
    "confidentialite": ["public", "interne", "confidentiel"],
    "date_minimale": "AAAA-MM-JJ",
}


def construire_self_query(
    question: str,
    *,
    client: Any | None = None,
    model: str | None = None,
) -> RequeteStructuree:
    """Demande à OpenAI un JSON séparant recherche sémantique et filtres."""
    if client is None:
        from openai import OpenAI

        client = OpenAI()
    model = model or os.getenv("OPENAI_MODEL")
    if not model:
        raise RuntimeError("Définissez OPENAI_MODEL.")
    reponse = client.responses.create(
        model=model,
        input=(
            "Retourne uniquement un objet JSON avec les clés texte et filtres. "
            f"Filtres autorisés : {json.dumps(SCHEMA, ensure_ascii=False)}. "
            f"Question : {question}"
        ),
    )
    donnees = json.loads(reponse.output_text)
    filtres = {cle: str(valeur) for cle, valeur in donnees.get("filtres", {}).items()}
    inconnus = set(filtres) - set(SCHEMA)
    if inconnus:
        raise ValueError(f"Filtres non autorisés : {sorted(inconnus)}")
    return RequeteStructuree(str(donnees["texte"]), filtres)


if __name__ == "__main__":
    if not os.getenv("OPENAI_API_KEY") or not os.getenv("OPENAI_MODEL"):
        print("Exemple prêt : définissez OPENAI_API_KEY et OPENAI_MODEL.")
    else:
        print(construire_self_query("Contrats juridiques publiés depuis 2025"))


## 8. Multi-Query

Générer des variantes puis fusionner les rangs avec RRF.

Script correspondant : [`08_multi_query.py`](examples/08_multi_query.py)

In [ ]:
# ruff: noqa: F811
"""Générer plusieurs formulations et fusionner leurs classements avec RRF."""

from __future__ import annotations

import os
from collections import defaultdict
from collections.abc import Callable, Hashable
from typing import Any, TypeVar

T = TypeVar("T", bound=Hashable)


def generer_variantes(
    question: str,
    nombre: int = 3,
    *,
    client: Any | None = None,
    model: str | None = None,
) -> list[str]:
    if client is None:
        from openai import OpenAI

        client = OpenAI()
    model = model or os.getenv("OPENAI_MODEL")
    if not model:
        raise RuntimeError("Définissez OPENAI_MODEL.")
    reponse = client.responses.create(
        model=model,
        input=f"Reformule cette question de {nombre} façons. Une formulation par ligne : {question}",
    )
    variantes = [ligne.strip(" -0123456789.") for ligne in reponse.output_text.splitlines()]
    return [question, *[variante for variante in variantes if variante][:nombre]]


def fusion_rrf(classements: list[list[T]], constante: int = 60) -> list[T]:
    scores: dict[T, float] = defaultdict(float)
    for classement in classements:
        for rang, element in enumerate(classement, start=1):
            scores[element] += 1 / (constante + rang)
    return sorted(scores, key=scores.get, reverse=True)


def multi_query(
    question: str,
    rechercher: Callable[[str], list[T]],
    *,
    client: Any | None = None,
    model: str | None = None,
) -> list[T]:
    requetes = generer_variantes(question, client=client, model=model)
    return fusion_rrf([rechercher(requete) for requete in requetes])


if __name__ == "__main__":
    if not os.getenv("OPENAI_API_KEY") or not os.getenv("OPENAI_MODEL"):
        print("Exemple prêt : définissez OPENAI_API_KEY et OPENAI_MODEL.")
    else:
        documents = ["retour sous 30 jours", "garantie de 24 mois", "remboursement"]
        recherche = lambda q: sorted(documents, key=lambda d: len(set(q.split()) & set(d.split())), reverse=True)
        print(multi_query("Quel est le délai de retour ?", recherche))


## 9. Ensemble BM25 + dense

Pondérer les signaux lexical et sémantique.

Script correspondant : [`09_ensemble.py`](examples/09_ensemble.py)

In [ ]:
# ruff: noqa: F811
"""Fusionner un classement BM25 et un classement dense."""

from __future__ import annotations

import re
from collections import defaultdict
from collections.abc import Callable

from rank_bm25 import BM25Okapi


def tokeniser(texte: str) -> list[str]:
    return re.findall(r"\w+", texte.lower())


def fusion_rrf_ponderee(
    classements: list[list[str]],
    poids: list[float],
    constante: int = 60,
) -> list[str]:
    if len(classements) != len(poids):
        raise ValueError("Un poids est requis par classement")
    scores: dict[str, float] = defaultdict(float)
    for classement, poids_source in zip(classements, poids, strict=True):
        for rang, document in enumerate(classement, start=1):
            scores[document] += poids_source / (constante + rang)
    return sorted(scores, key=scores.get, reverse=True)


def construire_ensemble(
    documents: list[str],
    recherche_dense: Callable[[str, int], list[str]],
    poids: tuple[float, float] = (0.4, 0.6),
):
    bm25 = BM25Okapi([tokeniser(document) for document in documents])

    def rechercher(question: str, k: int = 5) -> list[str]:
        scores = bm25.get_scores(tokeniser(question))
        lexical = [documents[i] for i in sorted(range(len(documents)), key=scores.__getitem__, reverse=True)[:k]]
        dense = recherche_dense(question, k)
        return fusion_rrf_ponderee([lexical, dense], list(poids))[:k]

    return rechercher


if __name__ == "__main__":
    corpus = ["retour sous trente jours", "garantie de deux ans", "livraison en cinq jours"]
    dense_demo = lambda question, k: sorted(corpus, key=lambda texte: abs(len(texte) - len(question)))[:k]
    moteur = construire_ensemble(corpus, dense_demo)
    print(moteur("délai pour retourner un produit", k=2))


## 10. Text-to-SQL

Limiter le modèle à une requête SELECT et à des tables autorisées.

Script correspondant : [`10_text_to_sql.py`](examples/10_text_to_sql.py)

In [ ]:
# ruff: noqa: F811
"""Générer une requête SQL avec OpenAI puis l'exécuter en lecture seule."""

from __future__ import annotations

import os
import re
import sqlite3
from typing import Any


def valider_select(requete: str, tables_autorisees: set[str]) -> str:
    """Accepte une seule instruction SELECT et un périmètre de tables explicite."""
    sql = requete.strip().removeprefix("```sql").removesuffix("```").strip().rstrip(";")
    if ";" in sql or not re.match(r"(?is)^select\b", sql):
        raise ValueError("Seule une instruction SELECT unique est autorisée")
    tables = set(re.findall(r"(?i)\b(?:from|join)\s+([a-z_][\w]*)", sql))
    if not tables or not tables <= tables_autorisees:
        raise ValueError(f"Table non autorisée : {sorted(tables - tables_autorisees)}")
    return sql


def question_vers_sql(
    question: str,
    schema: str,
    *,
    client: Any | None = None,
    model: str | None = None,
) -> str:
    if client is None:
        from openai import OpenAI

        client = OpenAI()
    model = model or os.getenv("OPENAI_MODEL")
    if not model:
        raise RuntimeError("Définissez OPENAI_MODEL.")
    reponse = client.responses.create(
        model=model,
        input=(
            "Produis une seule requête SQLite SELECT, sans commentaire ni Markdown. "
            f"Schéma : {schema}\nQuestion : {question}"
        ),
    )
    return reponse.output_text


def construire_text_to_sql(
    connexion: sqlite3.Connection,
    tables_autorisees: set[str],
    *,
    client: Any | None = None,
    model: str | None = None,
):
    schema = "\n".join(
        ligne[0]
        for ligne in connexion.execute(
            "SELECT sql FROM sqlite_master WHERE type='table' AND sql IS NOT NULL"
        )
    )

    def interroger(question: str) -> dict[str, object]:
        brute = question_vers_sql(question, schema, client=client, model=model)
        sql = valider_select(brute, tables_autorisees)
        return {"requete": sql, "resultat": connexion.execute(sql).fetchall()}

    return interroger


if __name__ == "__main__":
    if not os.getenv("OPENAI_API_KEY") or not os.getenv("OPENAI_MODEL"):
        print("Exemple prêt : définissez OPENAI_API_KEY et OPENAI_MODEL.")
    else:
        base = sqlite3.connect(":memory:")
        base.executescript("CREATE TABLE ventes(produit TEXT, montant REAL); INSERT INTO ventes VALUES ('A', 12.5), ('B', 7.5);")
        print(construire_text_to_sql(base, {"ventes"})("Quel est le total des ventes ?"))


## 11. Routeur RAG

Choisir entre documents, tables ou une réponse combinée.

Script correspondant : [`11_router_rag.py`](examples/11_router_rag.py)

In [ ]:
# ruff: noqa: F811
"""Orienter une question vers des documents, des tables ou les deux."""

from __future__ import annotations

import os
from enum import Enum
from typing import Any


class Source(Enum):
    DOCUMENTS = "documents"
    TABLES = "tables"
    LES_DEUX = "les_deux"


GABARIT = """Détermine où chercher la réponse.
- documents : procédures, politiques, explications, définitions
- tables : chiffres, comptages, agrégats, évolutions datées
- les_deux : un chiffre et son explication
Question : {question}
Réponds par un seul mot."""


def router(
    question: str,
    *,
    client: Any | None = None,
    model: str | None = None,
) -> Source:
    if client is None:
        from openai import OpenAI

        client = OpenAI()
    model = model or os.getenv("OPENAI_MODEL")
    if not model:
        raise RuntimeError("Définissez OPENAI_MODEL.")
    reponse = client.responses.create(model=model, input=GABARIT.format(question=question))
    try:
        return Source(reponse.output_text.strip().lower())
    except ValueError:
        return Source.DOCUMENTS


if __name__ == "__main__":
    if not os.getenv("OPENAI_API_KEY") or not os.getenv("OPENAI_MODEL"):
        print("Exemple prêt : définissez OPENAI_API_KEY et OPENAI_MODEL.")
    else:
        print(router("Combien de ventes ont été réalisées et pourquoi ?"))


## 12. Retriever de production

Filtrer, reclasser, journaliser et prévoir un repli.

Script correspondant : [`12_retriever_production.py`](examples/12_retriever_production.py)

In [ ]:
# ruff: noqa: F811
"""Pipeline de retrieval filtré, reclassé, journalisé et résilient."""

from __future__ import annotations

import logging
import time
from dataclasses import dataclass, field
from typing import Any

journal = logging.getLogger(__name__)


@dataclass
class Passage:
    texte: str
    source: str
    page: int
    score_dense: float = 0.0
    score_reclassement: float = 0.0
    metadonnees: dict[str, Any] = field(default_factory=dict)


class RetrieverProduction:
    def __init__(
        self,
        client,
        collection: str,
        encodeur,
        reclasseur=None,
        n_candidats: int = 20,
        n_final: int = 5,
        seuil_reclassement: float = 0.0,
    ) -> None:
        self.client = client
        self.collection = collection
        self.encodeur = encodeur
        self.reclasseur = reclasseur
        self.n_candidats = n_candidats
        self.n_final = n_final
        self.seuil = seuil_reclassement

    @classmethod
    def depuis_modeles(
        cls,
        url_qdrant: str,
        collection: str,
        modele_embedding: str,
        modele_reclassement: str = "cross-encoder/ms-marco-MiniLM-L6-v2",
    ) -> RetrieverProduction:
        from qdrant_client import QdrantClient
        from sentence_transformers import CrossEncoder, SentenceTransformer

        encodeur = SentenceTransformer(modele_embedding)
        try:
            reclasseur = CrossEncoder(modele_reclassement)
        except Exception as erreur:  # noqa: BLE001 - repli de disponibilité volontaire
            journal.warning("Reclasseur indisponible : %s", erreur)
            reclasseur = None
        return cls(QdrantClient(url=url_qdrant), collection, encodeur, reclasseur)

    def chercher(self, question: str, filtres: dict[str, object] | None = None) -> list[Passage]:
        depart = time.perf_counter()
        vecteur = self.encodeur.encode(question, normalize_embeddings=True).tolist()
        resultat = self.client.query_points(
            collection_name=self.collection,
            query=vecteur,
            query_filter=self._filtre(filtres),
            limit=self.n_candidats,
            with_payload=True,
            score_threshold=0.3,
        )
        passages = [
            Passage(
                texte=point.payload["texte"],
                source=point.payload.get("source", ""),
                page=point.payload.get("page", 0),
                score_dense=point.score,
                metadonnees=point.payload,
            )
            for point in resultat.points
        ]
        passages = self._reclasser(question, passages)
        journal.info(
            "retrieval | duree=%.0fms | candidats=%d | retenus=%d",
            (time.perf_counter() - depart) * 1000,
            len(resultat.points),
            min(len(passages), self.n_final),
        )
        return passages[: self.n_final]

    def _reclasser(self, question: str, passages: list[Passage]) -> list[Passage]:
        if self.reclasseur is None or len(passages) <= 1:
            return passages
        try:
            paires = [(question, passage.texte) for passage in passages]
            for passage, score in zip(passages, self.reclasseur.predict(paires), strict=True):
                passage.score_reclassement = float(score)
            retenus = [p for p in passages if p.score_reclassement >= self.seuil]
            return sorted(retenus, key=lambda p: p.score_reclassement, reverse=True)
        except Exception as erreur:  # noqa: BLE001 - repli dense volontaire
            journal.warning("Reclassement échoué, repli dense : %s", erreur)
            return passages

    @staticmethod
    def _filtre(filtres: dict[str, object] | None):
        if not filtres:
            return None
        from qdrant_client.models import FieldCondition, Filter, MatchValue

        return Filter(
            must=[
                FieldCondition(key=cle, match=MatchValue(value=valeur))
                for cle, valeur in filtres.items()
            ]
        )


if __name__ == "__main__":
    print("Injectez un client Qdrant et les modèles, ou utilisez RetrieverProduction.depuis_modeles().")


## 13. RAG itératif

Rechercher les informations manquantes avec une borne stricte.

Script correspondant : [`13_rag_iteratif.py`](examples/13_rag_iteratif.py)

In [ ]:
# ruff: noqa: F811
"""RAG itératif borné pour les questions nécessitant plusieurs recherches."""

from __future__ import annotations

import os
from typing import Any


def demander(client: Any, model: str, instruction: str) -> str:
    return client.responses.create(model=model, input=instruction).output_text.strip()


def rag_iteratif(
    question: str,
    retriever,
    max_iterations: int = 3,
    *,
    client: Any | None = None,
    model: str | None = None,
) -> dict[str, object]:
    if client is None:
        from openai import OpenAI

        client = OpenAI()
    model = model or os.getenv("OPENAI_MODEL")
    if not model:
        raise RuntimeError("Définissez OPENAI_MODEL.")

    contexte: list[Any] = []
    requete = question
    tours = 0
    for tours in range(1, max_iterations + 1):
        contexte.extend(retriever.invoke(requete))
        texte = "\n\n".join(document.page_content for document in contexte[:8])
        verdict = demander(
            client,
            model,
            f"Question : {question}\nContexte : {texte[:3000]}\n"
            "Réponds SUFFISANT ou MANQUE: <sous-question précise>.",
        )
        if verdict.startswith("SUFFISANT"):
            break
        if verdict.startswith("MANQUE:"):
            requete = verdict.split(":", 1)[1].strip()
        else:
            break
    texte = "\n\n".join(document.page_content for document in contexte[:8])
    reponse = demander(
        client,
        model,
        f"Contexte : {texte[:4000]}\nQuestion : {question}\nRéponds avec les sources.",
    )
    return {"reponse": reponse, "tours": tours, "sources": contexte}


if __name__ == "__main__":
    if not os.getenv("OPENAI_API_KEY") or not os.getenv("OPENAI_MODEL"):
        print("Exemple prêt : définissez OPENAI_API_KEY et OPENAI_MODEL.")


## 14. RAG adaptatif

Décider quand chercher, répondre ou demander une clarification.

Script correspondant : [`14_rag_adaptatif.py`](examples/14_rag_adaptatif.py)

In [ ]:
# ruff: noqa: F811
"""Décider s'il faut chercher, répondre directement ou clarifier."""

from __future__ import annotations

import os
from enum import Enum
from typing import Any


class Decision(Enum):
    CHERCHER = "chercher"
    REPONDRE = "repondre"
    CLARIFIER = "clarifier"


def decider(
    question: str,
    *,
    client: Any | None = None,
    model: str | None = None,
) -> Decision:
    if client is None:
        from openai import OpenAI

        client = OpenAI()
    model = model or os.getenv("OPENAI_MODEL")
    if not model:
        raise RuntimeError("Définissez OPENAI_MODEL.")
    reponse = client.responses.create(
        model=model,
        input=(
            f"Question : {question}\n"
            "Réponds par chercher pour des données internes/récentes, repondre pour un concept "
            "stable, ou clarifier si la question est ambiguë. Un seul mot."
        ),
    )
    try:
        return Decision(reponse.output_text.strip().lower())
    except ValueError:
        return Decision.CHERCHER


if __name__ == "__main__":
    if not os.getenv("OPENAI_API_KEY") or not os.getenv("OPENAI_MODEL"):
        print("Exemple prêt : définissez OPENAI_API_KEY et OPENAI_MODEL.")
    else:
        print(decider("Quel est le chiffre d'affaires interne de cette année ?"))


## Bilan

Le choix d'une base vectorielle dépend du volume, des filtres, de la latence, de l'exploitation et du coût. Commencez par une mesure locale, puis ajoutez les services et architectures avancées uniquement lorsqu'ils améliorent vos évaluations.